In [43]:
from datasets import load_dataset
import pickle

data_files = {
    "train": "NgramDataset/train-00000-of-00001.parquet",
    "validation": "NgramDataset/validation-00000-of-00001.parquet",
    "test": "NgramDataset/test-00000-of-00001.parquet",
}

dataset = load_dataset("parquet", data_files=data_files)
print(dataset["train"]["text"][:5])

['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgi

In [44]:
import string


def preprocess_text(dataset, n=2):
    cleaned_text = []
    for line in dataset:
        line = line.strip()

        if not line or line.startswith("="):
            continue

        line = line.lower().translate(str.maketrans("", "", string.punctuation))
        tokens = line.split()
        if tokens:
            start_padding = ["<start>"] * (n - 1)
            padded_tokens = start_padding + tokens + ["<end>"]
            cleaned_text.append(padded_tokens)
    return cleaned_text

In [45]:
from collections import Counter


# Build n-gram and context counts
def build_ngram(cleaned_text, n=2):
    ngram_counts = Counter()
    context_counts = Counter()
    vocab = set()

    for tokens in cleaned_text:
        # Add these list of tokens to vocabulary
        vocab.update(tokens)

        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i : i + n])
            context = tuple(tokens[i : i + n - 1])

            ngram_counts[ngram] += 1
            context_counts[context] += 1

    return ngram_counts, context_counts, vocab

In [46]:
def get_probability(n_gram, context, ngram_counts, context_counts, vocab_size):
    # Calculate probability with Laplace smoothing
    ngram_count = ngram_counts.get(n_gram, 0) + 1
    context_count = context_counts.get(context, 0) + vocab_size

    return ngram_count / context_count

## Calculating Preplexity
We calculate preplexity using logarithms rather than a chain of multiplication to avoid numerical underflow.
$$
PP = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i|w_{i-1})\right)
$$

In [47]:
import math


def get_preplexity(test_tokens, ngram_counts, context_counts, vocab_size, n):
    log_prob_sum = 0
    total_words = 0
    for tokens in test_tokens:
        for i in range(len(tokens) - n + 1):
            ngram = tuple(tokens[i : i + n])
            context = tuple(tokens[i : i + n - 1])

            prob = get_probability(
                ngram, context, ngram_counts, context_counts, vocab_size
            )

            log_prob_sum += math.log(prob)
            total_words += 1

    avg_log_prob = log_prob_sum / total_words
    preplexity = math.exp(-avg_log_prob)
    return preplexity

In [48]:
train = dataset["train"]["text"]
valid = dataset["validation"]["text"]
test = dataset["test"]["text"]

# Validation between unigram, bigram and trigram models
best_n = 1
lowest_preplexity = float("inf")
best_model_parameters = {}
model_parameters = []
for n in range(1, 4):
    cleaned_train = preprocess_text(train, n)
    cleaned_valid = preprocess_text(valid, n)

    # Train
    ngram_counts, context_counts, vocab = build_ngram(cleaned_train, n)
    vocab_size = len(vocab)
    model_parameters.append(
        {"ngram_counts": ngram_counts, "context_counts": context_counts, "vocab": vocab}
    )

    # Validate
    val_preplexity = get_preplexity(
        cleaned_valid, ngram_counts, context_counts, vocab_size, n
    )
    print(f"{n}-gram model validation perplexity: {val_preplexity:.3f}")

    if val_preplexity < lowest_preplexity:
        lowest_preplexity = val_preplexity
        best_n = n
        best_model_parameters = {
            "ngram_counts": ngram_counts,
            "context_counts": context_counts,
            "vocab_size": vocab_size,
        }

with open("ngram_models.pkl", "wb") as f:
    pickle.dump(model_parameters, f)

1-gram model validation perplexity: 2114.479
2-gram model validation perplexity: 9093.174
3-gram model validation perplexity: 39922.151


In [49]:
cleaned_test = preprocess_text(test, best_n)
test_preplexity = get_preplexity(
    cleaned_test,
    best_model_parameters["ngram_counts"],
    best_model_parameters["context_counts"],
    best_model_parameters["vocab_size"],
    best_n,
)

print(f"Best model: {best_n}-gram with test perplexity: {test_preplexity:.3f}")

Best model: 1-gram with test perplexity: 2231.861


In [50]:
import random


def generate(prompt, ngram_counts, context_counts, vocabulary, n, max_length=20):
    # Preprocess prompt
    tokens = prompt.lower().translate(str.maketrans("", "", string.punctuation)).split()
    start_padding = ["<start>"] * (n - 1)
    padded_tokens = start_padding + tokens

    vocab = list(vocabulary)
    vocab_size = len(vocab)

    for _ in range(max_length):
        context = tuple(padded_tokens[-(n - 1) :])
        probabilities = []

        # Calculate probabilities for each word in the vocabulary
        for word in vocab:
            ngram = context + (word,)
            probability = get_probability(
                ngram, context, ngram_counts, context_counts, vocab_size
            )
            probabilities.append(probability)

        # Sample next word based on probabilities
        next_word = random.choices(vocab, weights=probabilities, k=1)[0]
        if next_word == "<end>":
            break
        padded_tokens.append(next_word)

    return " ".join(padded_tokens[n - 1 :])

In [ ]:
# Load models from disk
with open("ngram_models.pkl", "rb") as f:
    loaded_models = pickle.load(f)

prompt = input("Enter a prompt:")

for i, model in enumerate(loaded_models):
    output = generate(
        prompt,
        model["ngram_counts"],
        model["context_counts"], 
        model["vocab"],
        i + 1
    )

    print(f"{i + 1}-gram Output: {output}")

1-gram Output: the cat is dhanno klm rancid porth helipad astonishing desecration construed vagrant mykolaiv harlequin stimi fiftieth ling smythe neolamarckia അള wherewith kipling nicholaievitch
2-gram Output: the cat is 1703 chandrabindu composted archway wallingford pilasters jinan cyberwarfare rogue 927 flamingos thursday nourish customize improvised ischium 84th pulled shaar interpretive
3-gram Output: the cat is xuri unworthy roca 417 hersey camcorders incident allegorical chihuahua menon psychoactivity 136xe 269 polymerase grates advantages brú unreceptive rfc urocerus
